# M05 · XGBoost

Entrenamos un clasificador **XGBoost** para predecir el grupo NOVA (1 a 4) de un producto.

**Objetivo:** macro F1 lo más alto posible, con un gap entre train y validación **< 5 %**.

## Paso 1 · Carga de datos

Partimos del fichero limpio `off_nova_clean.csv.gz` (lo generó el EDA). **No lo modificamos nunca**: cualquier transformación propia de este modelo irá dentro de un pipeline más adelante.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("data/processed/off_nova_clean.csv.gz")
if not DATA.exists():  # Jupyter lanzado desde notebooks/
    DATA = Path("../data/processed/off_nova_clean.csv.gz")

df = pd.read_csv(DATA, dtype={"code": str})
df.shape

(48119, 35)

## Paso 2 · Separar train y test

La columna `split` ya viene marcada en el fichero (80 % train, 20 % test). La usamos tal cual para que todos los modelos del equipo comparen con los mismos datos.

Separamos **antes de explorar nada**. El test queda guardado y no lo miramos: solo se usa una vez, al final, en M08, para la nota final del modelo elegido.

In [5]:
train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()

In [6]:
print("train:", train.shape)
print("test:", test.shape)

train: (38495, 35)
test: (9624, 35)


## Paso 3 · Definir X, y y los grupos (solo train)

- `y` es lo que queremos predecir: `nova_group`.
- `X` son las variables con las que el modelo predice. Quitamos columnas que no son features:
  - `code`, `product_name`, `brands`, `ingredients_text`: identificadores o texto libre.
  - `split`: solo sirve para separar.
  - `nova_group`: es la respuesta; si la dejamos en `X`, el modelo hace trampa.
  - `grupo_producto`: lo guardamos aparte en `groups`. No es una feature, pero lo necesita la validación cruzada para que un mismo producto (mismo nombre y marca, varias filas casi iguales) no caiga a la vez en train y en validación. Sin eso, el score saldría inflado.

In [7]:
NOT_FEATURES = ["code", "product_name", "brands", "ingredients_text",
                "split", "nova_group", "grupo_producto"]

y = train["nova_group"]
groups = train["grupo_producto"]
X = train.drop(columns=NOT_FEATURES)

In [8]:
print("X:", X.shape)
print(list(X.columns))
print()
print(y.value_counts().sort_index())
print()
print(y.value_counts(normalize=True).sort_index().round(3))

X: (38495, 28)
['additives_n', 'pnns_groups_1', 'pnns_groups_2', 'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'fruits-vegetables-legumes_100g', 'sin_lista_ingredientes', 'tiene_aromas', 'tiene_edulcorantes', 'tiene_jarabes', 'tiene_almidon_modificado', 'tiene_grasa_hidrogenada', 'tiene_proteina_aislada', 'tiene_colorantes', 'tiene_emulsionantes', 'tiene_potenciadores', 'tiene_aceite_palma', 'tiene_conservantes', 'tiene_sal', 'tiene_azucar', 'tiene_aceite', 'n_ingredientes']

nova_group
1     4952
2     4308
3     7865
4    21370
Name: count, dtype: int64

nova_group
1    0.129
2    0.112
3    0.204
4    0.555
Name: proportion, dtype: float64


### Resultados

- `X` tiene 38.495 filas y 28 columnas: son los productos de train y las variables con las que el modelo va a predecir.
- Hay 4 clases NOVA. La clase 4 tiene 21.370 productos (55,5 %); las otras son mucho más pequeñas: NOVA 3 con 20,4 %, NOVA 1 con 12,9 % y NOVA 2 con 11,2 %.
- Las clases están **desbalanceadas**. Un modelo que dijera siempre "NOVA 4" acertaría más de la mitad de las veces sin aprender nada. Por eso la métrica principal es **macro F1** (que da el mismo peso a cada clase) y no accuracy, y por eso miraremos el recall de cada clase, sobre todo de NOVA 2, que es la más pequeña.

## Paso 4 · Preparar las columnas

XGBoost trabaja bien con los datos casi tal cual, así que aquí hay poco que transformar:
- **Valores vacíos (nulos): NO los rellenamos.** XGBoost los maneja solo: en cada división del árbol aprende a qué lado mandar los vacíos. Además, que falte un dato es información útil (por ejemplo, a `fiber_100g` le falta el 78 % en NOVA 2: aceites, miel, azúcar). Si lo rellenáramos con la mediana, perderíamos esa señal. *(Decisión de M05; difiere de la imputación con mediana de otros modelos que sí la necesitan.)*
- **Binarias (0/1)**: se dejan tal cual.
- **Categoría `pnns_groups_2`**: **no usamos One-Hot**. XGBoost trata las categorías internamente (`enable_categorical=True`), así que la columna se queda como una sola, con el tipo `category` de pandas. Así no creamos ~40 columnas y la importancia de variables (y SHAP) se lee por columna original. Usamos `pnns_groups_2` y no `pnns_groups_1` (decisión del equipo en el EDA).
- **No hacemos escalado ni log**: XGBoost usa árboles y no los necesita.
- Excluimos `tiene_grasa_hidrogenada` (solo el 0,5 % de los productos).

Como no calculamos nada a partir de los datos (ni medianas, ni medias), **no hay riesgo de fuga de datos** y no hace falta un `ColumnTransformer`: basta con elegir las columnas.

### 4.1 Grupos de columnas

Clasificamos las columnas en tres listas:

- `cat_cols`: la columna de texto con una categoría (`pnns_groups_2`).
- `num_cols`: las 11 columnas numéricas continuas (nutrientes incluida la fibra, aditivos, nº de ingredientes). Pueden tener nulos y los dejamos así.
- `bin_cols`: las columnas 0/1 (`tiene_*` y `sin_lista_ingredientes`). Excluimos `tiene_grasa_hidrogenada`.

Después nos quedamos solo con esas columnas en `X`. `pnns_groups_1` y `tiene_grasa_hidrogenada` no están en ninguna lista, así que desaparecen.

In [12]:
cat_cols = ["pnns_groups_2"]
num_cols = ["additives_n", "energy-kcal_100g", "fat_100g", "saturated-fat_100g",
            "carbohydrates_100g", "sugars_100g", "fiber_100g", "proteins_100g",
            "salt_100g", "fruits-vegetables-legumes_100g", "n_ingredientes"]
bin_cols = [c for c in X.columns
            if c.startswith("tiene_") and c != "tiene_grasa_hidrogenada"] + ["sin_lista_ingredientes"]

X = X[cat_cols + num_cols + bin_cols]

print(len(cat_cols), len(num_cols), len(bin_cols))
print(X.shape)

1 11 14
(38495, 26)


### Resultados

- Salen 1, 11 y 14 columnas en cada grupo: 26 columnas en total. Las otras 2 de las 28 de `X` (`pnns_groups_1` y `tiene_grasa_hidrogenada`) quedan fuera a propósito.

### 4.2 Columna categórica: tipo `category`

XGBoost puede tratar las categorías internamente (`enable_categorical=True`), así que `pnns_groups_2` se queda como **una sola columna**, pero tiene que tener el tipo `category` de pandas (no texto normal).

Ventajas: no creamos ~40 columnas y la importancia de variables (y SHAP) se lee por columna original.

Ojo: las categorías se definen con train. Habrá que reutilizar la misma lista al evaluar con el test (M08).

In [13]:
X["pnns_groups_2"] = X["pnns_groups_2"].astype("category")

X["pnns_groups_2"].dtype

CategoricalDtype(categories=['Alcoholic beverages', 'Appetizers',
                  'Artificially sweetened beverages', 'Baby foods',
                  'Baby milks', 'Biscuits and cakes', 'Bread',
                  'Breakfast cereals', 'Cereals', 'Cheese',
                  'Chocolate products', 'Dairy desserts',
                  'Dressings and sauces', 'Dried fruits', 'Eggs', 'Fats',
                  'Fish and seafood', 'Fruit juices', 'Fruit nectars',
                  'Fruits', 'Ice cream', 'Legumes', 'Meat', 'Milk and yogurt',
                  'Nuts', 'Offals', 'One-dish meals', 'Pastries',
                  'Pizza pies and quiches', 'Plant-based milk substitutes',
                  'Potatoes', 'Processed meat', 'Salty and fatty products',
                  'Sandwiches', 'Soups', 'Sweetened beverages', 'Sweets',
                  'Teas and herbal teas and coffees', 'Unsweetened beverages',
                  'Vegetables', 'Waters and flavored waters', 'unknown'],
, ordered=False,

### Resultados

- `pnns_groups_2` ahora es de tipo `category` con 42 categorías.

### 4.3 Comprobamos los nulos

Verificamos qué columnas tienen valores vacíos. Los dejamos tal cual: XGBoost los gestiona solo.

In [14]:
nulos = X.isna().sum()
nulos[nulos > 0].sort_values(ascending=False)

fiber_100g                        12928
fruits-vegetables-legumes_100g     4314
saturated-fat_100g                 1178
sugars_100g                        1082
salt_100g                          1079
fat_100g                            800
proteins_100g                       793
carbohydrates_100g                  789
energy-kcal_100g                    482
additives_n                           3
n_ingredientes                        2
dtype: int64

## Paso 5 · Etiquetas, pesos y modelo

### 5.1 Etiquetas de 0 a 3

XGBoost exige que las clases empiecen en 0, y las nuestras van de 1 a 4 (NOVA 1 a 4). Creamos `y_xgb = y - 1` (valores 0, 1, 2, 3). Guardamos `y` original para las métricas.

In [15]:
y_xgb = y - 1

y_xgb.value_counts().sort_index()

nova_group
0     4952
1     4308
2     7865
3    21370
Name: count, dtype: int64

### Resultados

- Las clases ahora son 0, 1, 2, 3 (= NOVA 1, 2, 3, 4), con los mismos conteos que antes. Por ejemplo, la clase 3 (NOVA 4) sigue teniendo 21.370 productos.

### 5.2 Pesos por producto

Como NOVA 4 es el 55 % de los datos, el modelo tiende a ignorar las clases pequeñas. Para evitarlo damos a cada producto un **peso** inversamente proporcional al tamaño de su clase: más peso a los NOVA 1 y 2 (pocos productos), menos a los NOVA 4 (muchos). Así un error en una clase pequeña "cuesta" más durante el entrenamiento.

`compute_sample_weight("balanced", y_xgb)` calcula un peso por cada producto. Después comprobamos el peso de cada clase.

In [16]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weight = compute_sample_weight("balanced", y_xgb)

pd.Series(sample_weight).groupby(y_xgb.values).first().round(2)

0    1.94
1    2.23
2    1.22
3    0.45
dtype: float64

### 5.3 Definir el `XGBClassifier`

Creamos el modelo con valores iniciales razonables (los ajustaremos con Optuna en el ticket M07):

- `objective="multi:softprob"`: clasificación de 4 clases; devuelve una probabilidad por clase.
- `n_estimators=300`: número de árboles. Cada árbol corrige los errores de los anteriores.
- `learning_rate=0.1`: cuánto corrige cada árbol. Más bajo = más lento pero menos overfitting.
- `max_depth=6`: profundidad máxima de cada árbol. Más profundo = más riesgo de overfitting.
- `tree_method="hist"` y `enable_categorical=True`: necesarios para que XGBoost use `pnns_groups_2` como categoría nativa.
- `random_state=42`: para que los resultados se puedan repetir.
- `n_jobs=-1`: usa todos los núcleos del procesador.

Esta celda solo crea el modelo; todavía no lo entrena.

In [17]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective="multi:softprob",
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
)
model

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklear

### Resultados

- El modelo está creado: se ven `objective='multi:softprob'` y `enable_categorical=True`. Los valores `None` son los parámetros que dejamos con el valor por defecto de XGBoost.
- Todavía **no está entrenado**: no ha visto ningún dato. Lo entrenaremos dentro de la validación cruzada en el Paso 6.